# MGE Recipe — student walkthrough

## Learning to Autolens / Module 09

---

**What MGE replaces.** A standard `al.lp.Sersic` light profile has 7 nonlinear parameters (centre × 2, intensity, R_eff, n, ell_comps × 2). For asymmetric / disky / bulge+disk morphologies, one Sersic isn't expressive enough — you'd need 14 nonlinear params (two Sersic components) just to capture the angular structure.

**What MGE provides.** A **basis** of N (typically 20-30) Gaussian profiles whose amplitudes are linearly inverted at zero non-linear cost. The Gaussian σ values are spread on a log grid; only the global ellipticity + (optionally) centre are nonlinear. **Net: 3-4 nonlinear params for arbitrary morphology**, plus N linear amplitudes solved for free.

| Light | Nonlinear params | Captures |
|---|---|---|
| Sersic | 7 | Single elliptical |
| Bulge+disk (2× Sersic) | 14 | Two ellipticals at different PAs |
| MGE 30-Gaussians | 3-4 | Arbitrary morphology |

**This recipe** demonstrates:
1. The 1-line factory call to build an MGE basis.
2. How to combine MGE for lens light + parametric mass + parametric source in one Collection.
3. When to use 1 basis vs 2 (`gaussian_per_basis`) — bulge + envelope.
4. Reading the linear amplitudes after the fit converges.

---

## Step 1 — the factory call

`al.model_util.mge_model_from(...)` is the one function you need. Standard signature for a galaxy bulge:

In [ ]:
import os
os.environ.setdefault("PYAUTOFIT_TEST_MODE", "1")
from pathlib import Path
import autofit as af
import autolens as al
from IPython.display import Markdown, display
%matplotlib inline

# Single-basis MGE bulge — for galaxies without a clear bulge+envelope split
bulge_simple = al.model_util.mge_model_from(
    mask_radius=2.5,
    total_gaussians=20,
    gaussian_per_basis=1,           # 1 ellipticity for all 20 Gaussians
    centre_prior_is_uniform=False,
    centre=(0.0, 0.0),
    centre_sigma=0.05,
)
print(f"Simple MGE bulge: {bulge_simple.prior_count} nonlinear params "
      f"(+20 linear amplitudes inverted)")

## Step 2 — when to use 2 bases (`gaussian_per_basis=2`)

**One basis** = one global ellipticity for the whole light profile. Works for a smooth single-component galaxy.

**Two bases** = two independent ellipticities. Use this when the light has:
- A **bulge + envelope** with different ellipticities (most ETGs).
- A **bulge + disk** at different PAs (disky galaxies — see `Examples/disky_spiral_lens/`).
- Any visible PA twist between inner and outer isophotes.

Two bases is the autolens 2026.4 default for the BCG in cluster-scale SLaM, because clusters always have envelope-vs-core PA mismatch.

In [ ]:
bulge_two_basis = al.model_util.mge_model_from(
    mask_radius=2.5,
    total_gaussians=30,
    gaussian_per_basis=2,           # 2 independent ellipticities (15 Gaussians each)
    centre_prior_is_uniform=False,
    centre=(0.0, 0.0),
    centre_sigma=0.05,
)
print(f"Bulge+envelope MGE: {bulge_two_basis.prior_count} nonlinear params "
      f"(+30 linear amplitudes — 2 ell_comps pairs, shared centre)")

## Step 3 — assemble a full lens model with MGE light + parametric mass + parametric source

MGE replaces `bulge=` only; the mass profile + source are unchanged. Here's a complete galaxy-scale model with MGE bulge + Iso mass + Sersic source — the recipe used in Module 09 + the `disky_spiral_lens` example.

In [ ]:
# Lens galaxy: MGE bulge + Isothermal mass + ExternalShear
mass = af.Model(al.mp.Isothermal)
mass.einstein_radius = af.UniformPrior(lower_limit=0.5, upper_limit=2.5)
shear = af.Model(al.mp.ExternalShear)
shear.gamma_1 = af.GaussianPrior(mean=0.0, sigma=0.05)
shear.gamma_2 = af.GaussianPrior(mean=0.0, sigma=0.05)
lens = af.Model(
    al.Galaxy, redshift=0.5,
    bulge=bulge_two_basis,           # MGE light
    mass=mass,                       # parametric mass (unchanged)
    shear=shear,
)

# Source: parametric SersicCore (typical default; pixelize for complex sources)
source_bulge = af.Model(al.lp.SersicCore)
source = af.Model(al.Galaxy, redshift=2.0, bulge=source_bulge)

model = af.Collection(galaxies=af.Collection(lens=lens, source=source))
print(f"Full MGE-light + Iso-mass + SersicCore-source model:")
print(f"  Total free params: {model.prior_count}")
print(f"  vs single-Sersic equivalent: ~{model.prior_count + 4} (MGE saves ~4)")
print(f"  vs bulge+disk (2× Sersic): ~{model.prior_count + 11} (MGE saves ~11)")

## Step 4 — read the linear amplitudes after the fit

After convergence, the linear amplitudes are stored in `result.max_log_likelihood_fit.tracer_linear_light_profiles_to_light_profiles`. Each Gaussian's amplitude is now a real number; sum them weighted by their footprints to get the integrated luminosity for use in scaling relations (Faber-Jackson, etc.).

Reference: `autolens_workspace_latest/scripts/imaging/features/scaling_relation/modeling.py` lines 199-208 has the exact recipe for converting MGE amplitudes to total luminosity.

In [ ]:
# Pseudo-code (would run with a real result):
# tracer = result.max_log_likelihood_fit.tracer_linear_light_profiles_to_light_profiles
# import numpy as np
# luminosity_per_gaussian = [
#     2 * np.pi * g.sigma**2 / g.axis_ratio() * g.intensity
#     for g in tracer.galaxies[0].bulge.profile_list
# ]
# total_luminosity = float(np.sum(luminosity_per_gaussian) / pixel_scale**2)
# print(f"Total MGE luminosity: {total_luminosity}")
print("Run the fit, then post-process tracer.galaxies[i].bulge.profile_list "
      "for the per-Gaussian intensities.")

## Step 5 — gotchas

1. **`mask_radius` matters.** The factory uses it to determine the σ-grid spread (smallest σ ≈ pixel scale; largest ≈ mask_radius / 4). Set it to your *actual* analysis mask radius so the basis covers the right scales.
2. **`centre_prior_is_uniform=False` for the lens galaxy.** Default `True` gives uniform centre prior over a wide range — fine when the lens is the only luminous galaxy, painful when you have multiple. Use `False` + tight Gaussian centre for known-position galaxies.
3. **Linear amplitudes can go negative.** The inversion has no positivity constraint by default. Negative source amplitudes mean the fit is pulling "flux" out of the source plane to fit noise. To enforce positivity, use `al.model_util.mge_positive_model_from(...)` (a different factory that uses a Bayesian-positive prior).
4. **MGE doesn't help if your source is genuinely simple.** A clean Sersic source fits faster and equally well with `al.lp.Sersic`. MGE is for *complex* light, not all light.
5. **Combine with linear light profiles for the SOURCE too**, when needed. `al.lp_linear.Sersic` is a Sersic with linearly-inverted intensity (no nonlinear `intensity` parameter). For a single-source-component fit this saves 1 nonlinear param. The compound_lens_zoo R5 model uses `lp.SersicCore` for sources but `lp.Sersic` (regular) for lens light — historical.

---

## See also

- [`09_mge_linear_light_profiles.ipynb`](09_mge_linear_light_profiles.ipynb) — the main module notebook with full theoretical treatment
- [`Modules/05_Pixelized_Source_Reconstructions/06_pixelization_recipe.ipynb`](../05_Pixelized_Source_Reconstructions/06_pixelization_recipe.ipynb) — sister recipe for pixelized sources
- [`Examples/compound_lens_zoo/03_slam_recipe.ipynb`](../../Examples/compound_lens_zoo/03_slam_recipe.ipynb) — sister recipe for SLaM staging (uses MGE throughout)
- [`Examples/disky_spiral_lens/01_disky_spiral_fit.ipynb`](../../Examples/disky_spiral_lens/01_disky_spiral_fit.ipynb) — bulge+disk vs single-Sersic Bayes-factor demo (the empirical case for MGE)
- `autolens_workspace_latest/scripts/imaging/features/multi_gaussian_expansion/modeling.py` — canonical PyAutoLens MGE example
- Cappellari (2002), MNRAS 333, 400 — original MGE paper